In [1]:
import numpy as np
from numpy import cos, sin
import matplotlib.pyplot as plt

# Settings

In [2]:
# Example Save Code
# np.savez(
#     "3dorbit",
#     state = state,
#     time = time,
#     control = control,
#     time_c = time_c,
# )
# As long as the save file has state and time, this code will work. Edit the name to match the file of interest. Note that this code in its current state assumes that the state is in the nondimensionalized rotating frame.

file_name = "3dorbit"
path_modifier = "Standard_Cycler_Generation/"
reference_name = path_modifier + file_name + ".npz"
orbit = np.load(reference_name)

In [3]:
# Export Options: Set the first element to one for an embedded animation, set the second element to one for an mp4 (requires ffmpeg), and/or set the third element to one for a gif. Setting all elements to zero will disable the generation of the corresponding animation which shortens the run time. The name will be used as the file name when generating the animations.

# Can set this differently to control exported file names
data_export_name = file_name + "_inertial"
video_export_name = file_name

# 2D Inertial Frame
inert_2d = [0, 0, 0]
name_inert_2d = f"{video_export_name}_inert_2d"

# 3D Inertial Frame
inert_3d = [0, 0, 0]
name_inert_3d = f"{video_export_name}_inert_3d"

# 2D Rotating Frame
rot_2d = [0, 0, 0]
name_rot_2d = f"{video_export_name}_rot_2d"

# 3D Rotating Frame
rot_3d = [0, 0, 0]
name_rot_3d = f"{video_export_name}_rot_3d"

# Set to 1 to export inertial states, time, and body positions. States are in km and km/s, time is in days, and the combined_state contains the Earth and Moon positions for each step, with the first three elements being the x, y, z positions of the Earth and the second three being the position coordinates for the Moon.
export = 0

In [4]:
mu = 1.2150584270572e-2

def t_to_day(t):
    return t*27.321661/2/np.pi

def day_to_t(t):
    return t*2*np.pi/27.321661

def x_to_dist(x):
    return x*384400

def unit_convert(x):
    x_new = np.zeros(np.shape(x))
    for i in range(np.shape(x)[1]):
        x_new[0:3,i] = x_to_dist(x[0:3,i])
        x_new[3:6,i] = day_to_t(x_to_dist(x[3:6,i]))/86400
    return x_new

def rotate_state(state, time_ang):
    state_inertial = np.zeros(np.shape(state))
    for i in range(np.shape(state)[1]):
        x, y, z, vx, vy, vz = state[:,i]
        theta = time_ang[i]
        state_inertial[:,i] = np.array([
            x*cos(theta) - y*sin(theta),
            x*sin(theta) + y*cos(theta),
            z,
            vx*cos(theta) - vy*sin(theta),
            vx*sin(theta) + vy*cos(theta),
            vz])
    return state_inertial

def rotate_state2(state, time_ang):
    state_inertial = np.zeros(np.shape(state))
    for i in range(np.shape(state)[1]):
        x, y, z, vx, vy, vz = state[:,i]
        theta = time_ang[i]
        state_inertial[:,i] = np.array([
            x*cos(theta) - y*sin(theta),
            x*sin(theta) + y*cos(theta),
            z,
            (vx - y)*cos(theta) - (vy + x)*sin(theta),
            (vx - y)*sin(theta) + (vy + x)*cos(theta),
            vz])
    return unit_convert(state_inertial)

In [5]:
state_rotating = unit_convert(orbit["state"])
xr, yr, zr, vxr, vyr, vzr = state_rotating
time = orbit["time"]
state = rotate_state2(orbit["state"], time)
x, y, z, vx, vy, vz = state
t = t_to_day(time)

In [6]:
earth_state_rotating = x_to_dist(np.array([-mu, 0, 0]))
moon_state_rotating = x_to_dist(np.array([1 - mu, 0, 0]))
combined_state_rotating = np.concatenate((earth_state_rotating, moon_state_rotating)).reshape(-1,1)*np.ones((6,np.shape(state)[1]))
exr, eyr, ezr, mxr, myr, mzr = combined_state_rotating
combined_state = rotate_state(combined_state_rotating, time)
ex, ey, ez, mx, my, mz = combined_state

In [7]:
if export == 1:
    np.savez(
        data_export_name,
        state = state,
        time = t,
        combined_state = combined_state
    )

In [8]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 250

from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import Circle
from IPython.display import HTML
from IPython.display import Video
from IPython.display import Image

## 2D Inertial

In [9]:
# Sets the resolution of the animation by only including frames every multiple of step. Higher numbers are smoother, lower numbers are faster to run. The animation speed is automatically scaled to keep the motion speed the same.
step = 10

# Sets the speed of the animation. Higher numbers are slower, lower numbers are faster.
speed = 40

In [10]:
# ChatGPT Special for Animation Code

fig, ax = plt.subplots(figsize=(8, 8))

# faint full trajectory in background
ax.plot(x, y, linewidth=1, alpha=0.4)

R_earth = 6378.137
R_moon  = 1738.139

# moving markers
earth_patch = Circle((ex[0], ey[0]), R_earth, label='Earth', color='Green')
moon_patch  = Circle((mx[0], my[0]), R_moon, label='Moon', color='Grey')
ax.add_patch(earth_patch)
ax.add_patch(moon_patch)
sat_point, = ax.plot([], [], 'o', markersize=0.1, label='Satellite', color='Black')
# earth_point, = ax.plot([], [], 'o', markersize=10, label='Earth')
# moon_point, = ax.plot([], [], 'o', markersize=8, label='Moon')

# optional trail behind spacecraft
trail_line, = ax.plot([], [], '-', linewidth=1)

ax.set_aspect('equal')
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_title('Earth-Moon Inertial Frame Animation (To Scale)')
ax.grid(True)
ax.legend()

# axis limits
x_all = np.concatenate([x, ex, mx])
y_all = np.concatenate([y, ey, my])
pad = 0.05 * max(x_all.max() - x_all.min(), y_all.max() - y_all.min())

ax.set_xlim(x_all.min() - pad, x_all.max() + pad)
ax.set_ylim(y_all.min() - pad, y_all.max() + pad)

time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

# trail_len = 100  # number of past points to show

def init():
    sat_point.set_data([], [])
    earth_patch.center = (ex[0], ey[0])
    moon_patch.center = (mx[0], my[0])
    trail_line.set_data([], [])
    time_text.set_text('')
    return sat_point, earth_patch, moon_patch, trail_line, time_text

def update(frame):
    sat_point.set_data([x[frame]], [y[frame]])
    earth_patch.center = (ex[frame], ey[frame])
    moon_patch.center = (mx[frame], my[frame])

    # start = max(0, frame - trail_len)
    start = 0
    trail_line.set_data(x[start:frame+1], y[start:frame+1])

    time_text.set_text(f't = {t[frame]:.3f}')
    return sat_point, earth_patch, moon_patch, trail_line, time_text

frames = range(0, len(t), step)

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    init_func=init,
    blit=True,
    interval=speed*step
) 

plt.close(fig)

if inert_2d[1] == 1:
    anim.save(name_inert_2d+".mp4", writer="ffmpeg", fps=20)
if inert_2d[2] == 1:
    anim.save(name_inert_2d+".gif", writer="ffmpeg", fps=20)

if inert_2d[0] == 1:
    display(HTML(anim.to_jshtml()))
elif inert_2d[1] == 1:
    display(Video(name_inert_2d+".mp4"))
elif inert_2d[2] == 1:
    display(Image(name_inert_2d+".gif"))

## 3D Inertial

In [11]:
# Sets the resolution of the animation by only including frames every multiple of step. Higher numbers are smoother, lower numbers are faster to run. The animation speed is automatically scaled to keep the motion speed the same.
step = 10

# Sets the speed of the animation. Higher numbers are slower, lower numbers are faster.
speed = 40

# Sets the camera angle.
elevation = 30
azimuth = -60

In [12]:
# ChatGPT Special Yet Again for Animation Code

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.view_init(elev=elevation, azim=azimuth)

# full trajectory (faint)
ax.plot(x, y, z, linewidth=1, alpha=0.3)

R_earth = 6378.137
R_moon  = 1738.139

# moving objects
sat_point, = ax.plot([], [], [], 'o', markersize=0.1, color='Black')
earth_point, = ax.plot([], [], [], 'o', markersize=5, color='Green')
moon_point, = ax.plot([], [], [], 'o', markersize=1, color='Grey')

# optional trail
trail_line, = ax.plot([], [], [], '-', linewidth=1)

proj1, = ax.plot([], [], [], zdir='z', color='black', alpha=0.1, linestyle='--')
proj2, = ax.plot([], [], [], zdir='y', color='black', alpha=0.1, linestyle='--')
proj3, = ax.plot([], [], [], zdir='x', color='black', alpha=0.1, linestyle='--')

ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.set_title('3D Earth-Moon Inertial Frame (Not Quite to Scale)')

# IMPORTANT: equal-ish scaling
max_range = max(
    (x.max()-x.min()),
    (y.max()-y.min()),
    (z.max()-z.min())
) / 2

mid_x = (x.max()+x.min()) / 2
mid_y = (y.max()+y.min()) / 2
mid_z = (z.max()+z.min()) / 2

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

zmin, zmax = ax.get_zlim()
ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()
ax.plot(x, y, zs=zmin, zdir='z', color='black', alpha=0.05, linestyle='--')
ax.plot(x, z, zs=ymax, zdir='y', color='black', alpha=0.05, linestyle='--')
ax.plot(y, z, zs=xmin, zdir='x', color='black', alpha=0.05, linestyle='--')

# trail_len = 50

def init():
    sat_point.set_data([], [])
    sat_point.set_3d_properties([])
    trail_line.set_data([], [])
    trail_line.set_3d_properties([])
    proj1.set_data([],[])
    proj2.set_data([],[])
    proj3.set_data([],[])
    proj1.set_3d_properties([])
    proj2.set_3d_properties([])
    proj3.set_3d_properties([])
    return sat_point, earth_point, moon_point, trail_line, proj1, proj2, proj3

def update(frame):
    # satellite
    sat_point.set_data([x[frame]], [y[frame]])
    sat_point.set_3d_properties([z[frame]])

    # Earth & Moon
    earth_point.set_data([ex[frame]], [ey[frame]])
    earth_point.set_3d_properties([ez[frame]])

    moon_point.set_data([mx[frame]], [my[frame]])
    moon_point.set_3d_properties([mz[frame]])

    # trail
    # start = max(0, frame - trail_len)
    start = 0
    trail_line.set_data(x[start:frame+1], y[start:frame+1])
    trail_line.set_3d_properties(z[start:frame+1])

    proj1.set_data(x[start:frame+1], y[start:frame+1])
    proj2.set_data(x[start:frame+1], z[start:frame+1])
    proj3.set_data(y[start:frame+1], z[start:frame+1])
    proj1.set_3d_properties(np.full_like(x[start:frame+1], zmin), zdir='z')
    proj2.set_3d_properties(np.full_like(x[start:frame+1], ymax), zdir='y')
    proj3.set_3d_properties(np.full_like(y[start:frame+1], xmin), zdir='x')

    return sat_point, earth_point, moon_point, trail_line, proj1, proj2, proj3

frames = range(0, len(t), step)

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    init_func=init,
    blit=False,   # ⚠️ must be False in 3D
    interval=speed*step
)

plt.close(fig)

if inert_3d[1] == 1:
    anim.save(name_inert_3d+".mp4", writer="ffmpeg", fps=20)
if inert_3d[2] == 1:
    anim.save(name_inert_3d+".gif", writer="ffmpeg", fps=20)

if inert_3d[0] == 1:
    display(HTML(anim.to_jshtml()))
elif inert_3d[1] == 1:
    display(Video(name_inert_3d+".mp4"))
elif inert_3d[2] == 1:
    display(Image(name_inert_3d+".gif"))

## 2D Rotating

In [13]:
# Sets the resolution of the animation by only including frames every multiple of step. Higher numbers are smoother, lower numbers are faster to run. The animation speed is automatically scaled to keep the motion speed the same.
step = 10

# Sets the speed of the animation. Higher numbers are slower, lower numbers are faster.
speed = 40

In [14]:
fig, ax = plt.subplots(figsize=(8, 8))

# faint full trajectory in background
ax.plot(xr, yr, linewidth=1, alpha=0.4)

R_earth = 6378.137
R_moon  = 1738.139

# moving markers
earth_patch = Circle((exr[0], eyr[0]), R_earth, label='Earth', color='Green')
moon_patch  = Circle((mxr[0], myr[0]), R_moon, label='Moon', color='Grey')
ax.add_patch(earth_patch)
ax.add_patch(moon_patch)
sat_point, = ax.plot([], [], 'o', markersize=0.1, label='Satellite', color='Black')
# earth_point, = ax.plot([], [], 'o', markersize=10, label='Earth')
# moon_point, = ax.plot([], [], 'o', markersize=8, label='Moon')

# optional trail behind spacecraft
trail_line, = ax.plot([], [], '-', linewidth=1)

ax.set_aspect('equal')
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_title('Earth-Moon Rotating Frame Animation (To Scale)')
ax.grid(True)
ax.legend()

# axis limits
x_all = np.concatenate([xr, exr, mxr])
y_all = np.concatenate([yr, eyr, myr])
pad = 0.05 * max(x_all.max() - x_all.min(), y_all.max() - y_all.min())

ax.set_xlim(x_all.min() - pad, x_all.max() + pad)
ax.set_ylim(y_all.min() - pad, y_all.max() + pad)

time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

# trail_len = 100  # number of past points to show

def init():
    sat_point.set_data([], [])
    earth_patch.center = (exr[0], eyr[0])
    moon_patch.center = (mxr[0], myr[0])
    trail_line.set_data([], [])
    time_text.set_text('')
    return sat_point, earth_patch, moon_patch, trail_line, time_text

def update(frame):
    sat_point.set_data([xr[frame]], [yr[frame]])
    earth_patch.center = (exr[frame], eyr[frame])
    moon_patch.center = (mxr[frame], myr[frame])

    # start = max(0, frame - trail_len)
    start = 0
    trail_line.set_data(xr[start:frame+1], yr[start:frame+1])

    time_text.set_text(f't = {t[frame]:.3f}')
    return sat_point, earth_patch, moon_patch, trail_line, time_text

frames = range(0, len(t), step)

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    init_func=init,
    blit=True,
    interval=speed*step
) 

plt.close(fig)

if rot_2d[1] == 1:
    anim.save(name_rot_2d+".mp4", writer="ffmpeg", fps=20)
if rot_2d[2] == 1:
    anim.save(name_rot_2d+".gif", writer="ffmpeg", fps=20)

if rot_2d[0] == 1:
    display(HTML(anim.to_jshtml()))
elif rot_2d[1] == 1:
    display(Video(name_rot_2d+".mp4"))
elif rot_2d[2] == 1:
    display(Image(name_rot_2d+".gif"))

## 3D Rotating

In [15]:
# Sets the resolution of the animation by only including frames every multiple of step. Higher numbers are smoother, lower numbers are faster to run. The animation speed is automatically scaled to keep the motion speed the same.
step = 10

# Sets the speed of the animation. Higher numbers are slower, lower numbers are faster.
speed = 40

# Sets the camera angle.
elevation = 30
azimuth = -60

In [16]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.view_init(elev=elevation, azim=azimuth)

# full trajectory (faint)
ax.plot(xr, yr, zr, linewidth=1, alpha=0.3)

R_earth = 6378.137
R_moon  = 1738.139

# moving objects
sat_point, = ax.plot([], [], [], 'o', markersize=0.1, color='Black')
earth_point, = ax.plot([], [], [], 'o', markersize=5, color='Green')
moon_point, = ax.plot([], [], [], 'o', markersize=1, color='Grey')

# optional trail
trail_line, = ax.plot([], [], [], '-', linewidth=1)

proj1, = ax.plot([], [], [], zdir='z', color='black', alpha=0.1, linestyle='--')
proj2, = ax.plot([], [], [], zdir='y', color='black', alpha=0.1, linestyle='--')
proj3, = ax.plot([], [], [], zdir='x', color='black', alpha=0.1, linestyle='--')

ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.set_title('3D Earth-Moon Inertial Frame (Not Quite to Scale)')

# IMPORTANT: equal-ish scaling
max_range = max(
    (xr.max()-xr.min()),
    (yr.max()-yr.min()),
    (zr.max()-zr.min())
) / 2

mid_x = (xr.max()+xr.min()) / 2
mid_y = (yr.max()+yr.min()) / 2
mid_z = (zr.max()+zr.min()) / 2

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

zmin, zmax = ax.get_zlim()
ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()
ax.plot(xr, yr, zs=zmin, zdir='z', color='black', alpha=0.05, linestyle='--')
ax.plot(xr, zr, zs=ymax, zdir='y', color='black', alpha=0.05, linestyle='--')
ax.plot(yr, zr, zs=xmin, zdir='x', color='black', alpha=0.05, linestyle='--')

# trail_len = 50

def init():
    sat_point.set_data([], [])
    sat_point.set_3d_properties([])
    trail_line.set_data([], [])
    trail_line.set_3d_properties([])
    proj1.set_data([],[])
    proj2.set_data([],[])
    proj3.set_data([],[])
    proj1.set_3d_properties([])
    proj2.set_3d_properties([])
    proj3.set_3d_properties([])
    return sat_point, earth_point, moon_point, trail_line

def update(frame):
    # satellite
    sat_point.set_data([xr[frame]], [yr[frame]])
    sat_point.set_3d_properties([zr[frame]])

    # Earth & Moon
    earth_point.set_data([exr[frame]], [eyr[frame]])
    earth_point.set_3d_properties([ezr[frame]])

    moon_point.set_data([mxr[frame]], [myr[frame]])
    moon_point.set_3d_properties([mzr[frame]])

    # trail
    # start = max(0, frame - trail_len)
    start = 0
    trail_line.set_data(xr[start:frame+1], yr[start:frame+1])
    trail_line.set_3d_properties(zr[start:frame+1])

    proj1.set_data(xr[start:frame+1], yr[start:frame+1])
    proj2.set_data(xr[start:frame+1], zr[start:frame+1])
    proj3.set_data(yr[start:frame+1], zr[start:frame+1])
    proj1.set_3d_properties(np.full_like(xr[start:frame+1], zmin), zdir='z')
    proj2.set_3d_properties(np.full_like(xr[start:frame+1], ymax), zdir='y')
    proj3.set_3d_properties(np.full_like(yr[start:frame+1], xmin), zdir='x')

    return sat_point, earth_point, moon_point, trail_line

frames = range(0, len(t), step)

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    init_func=init,
    blit=False,   # ⚠️ must be False in 3D
    interval=speed*step
)

plt.close(fig)

if rot_3d[1] == 1:
    anim.save(name_rot_3d+".mp4", writer="ffmpeg", fps=20)
if rot_3d[2] == 1:
    anim.save(name_rot_3d+".gif", writer="ffmpeg", fps=20)

if rot_3d[0] == 1:
    display(HTML(anim.to_jshtml()))
elif rot_3d[1] == 1:
    display(Video(name_rot_3d+".mp4"))
elif rot_3d[2] == 1:
    display(Image(name_rot_3d+".gif"))